# Experimento: FT-Transformer, fase 6 (mineração de features)

**Por que esta fase.** O que mais subiu o KS nas duas redes foram as **features**: o `tenure_1` no MLP e as três features do MLP no FT (fase 4, +0,0155, a primeira vitória do FT). Na fase 5, o Optuna encontrou uma rede diferente da base antiga: `o5_top3`, com 1 bloco, dropout de atenção de ~0,49, sem oversampling e ~55 épocas. Ela chegou a **0,5562** no placar e tem o maior KS OOF do projeto (0,5508). As features precisam ser medidas de novo **nessa** rede.

## De onde vêm as candidatas

**Análise de erro nas OOF do `o5_top3`.** Uma árvore rasa no resíduo (y − p) mostrou o que a rede não capturou:

| O que a árvore encontrou | Churn real | Prob. da rede |
|---|---|---|
| mensal > 98, razão de cobrança > 1,01, sem suporte técnico | 0,556 | 0,397 |
| mensal > 98, razão ≤ 1,01, sem backup | 0,428 | 0,346 |
| mensal entre 59 e 98, razão ≤ 0,96 | 0,377 | 0,501 |

A razão de cobrança é `TotalCharges / (MonthlyCharges × tenure)`. Quando ela fica abaixo de 1, o preço atual está acima da média que o cliente já pagou. Além disso, o **KS dentro do contrato mês a mês é 0,395**, o pior entre os segmentos, justamente onde estão 43% dos que saem.

| Candidata | Definição |
|---|---|
| `cobranca_razao` | TotalCharges / (MonthlyCharges × max(tenure, 1)) |
| `preco_subiu` | razão < 1 |
| `caro_sem_suporte` / `caro_sem_backup` | mensal > 98 e sem suporte técnico / sem backup |
| `mtm_tenure` | tenure, só para o contrato mês a mês (0 nos outros) |
| `mtm_sem_suporte` / `mtm_eletronico` | mês a mês sem suporte técnico / pagando com cheque eletrônico |
| `antigo_mtm` / `antigo` | mês a mês com mais de 24 meses / mais de 48 meses de casa |
| `muitos_servicos` | 5 ou mais serviços extras |
| as 9 candidatas do MLP | testadas na fase 5 só sobre a base antiga |

## Seleção para frente

Cada rodada mede a base atual e cada candidata somada a ela, nas **divisões próprias do Optuna** (sementes 201 a 203, 15 dobras), e não nas do placar. Entra a candidata de maior ganho no KS OOF, desde que ganhe em pelo menos 80% das reamostragens do bootstrap pareado. A seleção para quando nenhuma passa, ou depois de `RODADAS` rodadas. Só o conjunto final vai para o placar: sozinho e com a média de 3 redes, contra o `o5_top3`, contra o melhor FT (`o5_top1_3redes`), contra o trial 0 e contra o MLP adotado.

Todas as features são calculadas por linha, sem olhar o rótulo. As regras vieram da análise do treino, então só o veredito do placar e, no fim, o teste usado uma vez dizem se elas valem.

## Como executar no Kaggle

1. *File → Import Notebook* → `experimentos/ft_transformer/ft_transformer_fase6.ipynb`, com o dataset anexado.
2. *Add-ons → Secrets:* `GITHUB_TOKEN` e `WANDB_API_KEY`. **Nunca escreva os tokens neste notebook.**
3. *Settings:* Internet **On** e acelerador **GPU T4 x2**.
4. Rode com `MODO_TESTE = True`. Depois, com `False`, use *Save Version → Save & Run All (Commit)*. São cerca de 4 h. Para continuar uma execução que caiu, use `RUN_NAME` com o mesmo nome e `RESUME_FROM = "github"`: as medidas já feitas não se repetem.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/ft-transformer"           # branch com o código (precisa estar publicada no GitHub)
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "ft-fase6-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = "/kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv"   # vazio = procurar no repositório e em /kaggle/input
POR_GPU = 2                      # processos (dobras ao mesmo tempo) em cada GPU; sem GPU, processos na CPU
FASE5_RUN = "ft-fase5-20261117-1830"   # execução da fase 5 na branch de resultados (ou uma pasta local): traz as fases 1 a 5 já medidas
RODADAS = 3                     # rodadas da seleção para frente (cada uma mede todas as candidatas que sobraram)
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa configurações novas e encerra limpo (o Kaggle corta em 12 h)

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "ft_transformer" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

In [ ]:
import json

PASTA_FT = REPO_DIR / "experimentos" / "ft_transformer"
sys.path[:0] = [str(REPO_DIR), str(PASTA_FT)]
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), str(PASTA_FT), os.environ.get("PYTHONPATH", "")])  # processos paralelos acham os módulos
os.environ.setdefault("TORCH_DISABLE_NATIVE_JIT", "1")
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("ft-fase6-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "3"          # antes de importar modelo.py, que lê o limite na importação
    RODADAS = 1
    print("MODO_TESTE: 3 épocas, duas candidatas e uma rodada. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def continuar():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO) > 0


import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy.stats import ks_2samp
import torch
import comum, modelo as M, fase1 as F1, fase2 as F2, fase4 as F4, fase5 as F5, fase6 as F6

DISPOSITIVOS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
if DISPOSITIVOS == ["cpu"]:
    POR_GPU = os.cpu_count() or 1                 # sem GPU: um processo por núcleo
pd.set_option("display.max_columns", None, "display.width", 220, "display.max_colwidth", 70, "display.max_rows", 80)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | dispositivos: {DISPOSITIVOS} x {POR_GPU} processos | até {M.MAX_EPOCAS} épocas")

# O envio ao GitHub é testado antes de qualquer treino
json.dump(dict(run=RUN_NAME, inicio=time.strftime("%Y-%m-%d %H:%M")), open(os.path.join(PASTA, "inicio.json"), "w"))
GITHUB_OK = bool(RESULTS_REPO_URL and GITHUB_TOKEN) and salvar()
if RESULTS_REPO_URL and not GITHUB_OK:
    print("=" * 90 + "\nATENÇÃO: o envio ao GitHub falhou. Troque o GITHUB_TOKEN nos Secrets ou baixe a pasta de resultados no fim.\n" + "=" * 90)
else:
    print("GitHub:", "envio testado e funcionando" if GITHUB_OK else "desligado")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="ft_transformer", config=dict(modelo="ft_transformer", fase=6, dispositivos=DISPOSITIVOS,
                                                               por_gpu=POR_GPU, teste=MODO_TESTE, padrao=M.PADRAO))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def ao_registrar(r):
    """Cada configuração medida vai para o W&B e para o GitHub na hora: se o Kaggle cair, nada se perde."""
    if wandb_run is not None:
        v = r.get("vs_trial0") or {}
        wandb_run.log(dict(rotulo=r["rotulo"], ks_placar=r["ks_kfold"], ep=r["ep"], ks_oof=r["ks_oof"], auc_oof=r["auc_oof"],
                           mse_treino=r.get("mse_treino"), mse_val=r.get("mse_val"), epocas=r.get("epocas"),
                           minutos=r.get("tempo_min"), delta_vs_trial0=v.get("delta")))
    salvar()


# As linhas das fases 1 a 5 (com o trial 0, o o5_top3 e o o5_top1_3redes) entram aqui sem treinar de novo
import tempfile
if FASE5_RUN and not os.path.isfile(os.path.join(PASTA, "placar.jsonl")):
    with tempfile.TemporaryDirectory() as _tmp:
        ok = (shutil.copytree(FASE5_RUN, _tmp, dirs_exist_ok=True) and True) if os.path.isdir(FASE5_RUN) else \
            fluxo.baixar_github(_tmp, FASE5_RUN, RESULTS_REPO_URL or REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
        if ok and os.path.isfile(os.path.join(_tmp, "placar.jsonl")):
            shutil.copy(os.path.join(_tmp, "placar.jsonl"), PASTA)
            shutil.copytree(os.path.join(_tmp, "placar_oof"), os.path.join(PASTA, "placar_oof"), dirs_exist_ok=True)
anteriores = {r["rotulo"]: r for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer"}
print(f"Fases 1 a 5: {len(anteriores)} linhas trazidas de {FASE5_RUN}" if anteriores else "Fases 1 a 5: nada trazido; as bases serão medidas aqui")


def medir(itens):
    return F1.garantir(PASTA, itens, DISPOSITIVOS, POR_GPU, continuar=continuar, ao_registrar=ao_registrar)


def mostrar(linhas, campos=("vs_trial0", "vs_base")):
    display(F1.tabela(linhas, campos).round(4))

In [ ]:
# Verificações rápidas antes de gastar tempo de treino
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis com os outros modelos"
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
D = F1.divs()
assert len(D) == 15
for d in D:
    assert not set(d["treino"]) & set(d["validacao"]) and not set(d["ajuste"]) & set(d["parada"])
Xtr_, _, ytr_, _ = F1._dados()
Xa_, _, _ = comum.preparar(Xtr_, ytr_, Xtr_.iloc[:5])
n_param = sum(p.numel() for p in M.FTTransformer(Xa_.shape[1], **M.arquitetura(M.PADRAO)).parameters())
print(f"teste com a assinatura oficial | KS conferido contra scipy | 15 divisões sem sobreposição | assinatura do treino {F1.assinatura()}")
print(f"{Xa_.shape[1]} features = {Xa_.shape[1]} tokens + [CLS] | FT-Transformer padrão: {n_param:,} parâmetros para {len(Xtr_):,} clientes")
Xf_, _, _ = comum.preparar(Xtr_, ytr_, Xtr_.iloc[:5], features=F4.TRES)
print(f"com as features do MLP: {Xf_.shape[1]} tokens + [CLS] ({', '.join(F4.TRES)})")
for nome, hp in (("tokens categóricos", dict(tokens="cat")), ("PLR", dict(num_emb="plr"))):
    g_, n_ = M.estrutura(hp, list(Xf_.columns), np.asarray(Xf_, dtype="float32"))
    print(f"{nome}: {len(g_)} tokens | colunas com PLR: {[Xf_.columns[j] for j in n_]}")
Xn_ = comum.adicionar_features(Xtr_, F6.NOVAS)
y_ = np.asarray(ytr_)
for f in F6.NOVAS:
    v = Xn_[f]
    print(f"{f:<18}", f"{int(v.sum()):>5} clientes, churn {y_[v == 1].mean():.3f} contra {y_[v == 0].mean():.3f}" if v.nunique() <= 2
          else f"numérica, correlação com o churn {np.corrcoef(v, y_)[0, 1]:+.3f}")

# 1. Seleção para frente nas divisões do Optuna

Cada linha é uma candidata somada à base da rodada. `delta_oof` é o ganho no KS OOF contra essa base, e `p_positivo` é a fração das reamostragens do bootstrap com ganho. Entra a de maior `delta_oof`, se `p_positivo` ≥ 0,8.

In [ ]:
assert MODO_TESTE or (F6.ROT_BASE in anteriores and F1._chave(anteriores[F6.ROT_BASE]["config"]) == F1._chave(F6.BASE)), \
    "o o5_top3 trazido da fase 5 não bate com fase6.BASE"
CAND = ("mtm_eletronico", "caro_sem_suporte") if MODO_TESTE else F6.CANDIDATAS
D_SEL = F5.divs_optuna(F5.SEMENTES_OPTUNA[:1] if MODO_TESTE else F5.SEMENTES_OPTUNA)
y_tr = np.asarray(F1._dados()[2])


def ao_medir(l):
    print(f"  rodada {l['rodada']} | {l['candidata']:<20} KS {l['ks_kfold']:.4f} (base {l['ks_kfold_base']:.4f}) | "
          f"delta OOF {l['delta_oof']:+.4f} p+ {l['p_positivo']:.2f}", flush=True)
    if wandb_run is not None:
        wandb_run.log({"selecao/rodada": l["rodada"], "selecao/delta_oof": l["delta_oof"], "selecao/p_positivo": l["p_positivo"]})
    salvar()


t0 = time.time()
escolhidas, hist = F6.selecao_para_frente(PASTA, lambda c: F1.medir(c, DISPOSITIVOS, POR_GPU, D=D_SEL), y_tr,
                                          candidatas=CAND, rodadas=RODADAS, continuar=continuar, ao_medir=ao_medir)
print(f"{(time.time() - t0) / 60:.1f} min | escolhidas: {escolhidas or 'nenhuma'}")
json.dump(dict(escolhidas=escolhidas, hist=hist), open(os.path.join(PASTA, "selecao.json"), "w"), indent=1)
th = pd.DataFrame(hist)
if len(th):
    th["ic95"] = th.ic95.map(lambda v: f"[{v[0]:+.4f}, {v[1]:+.4f}]")
    for rd, g in th.groupby("rodada"):
        print(f"Rodada {rd}")
        display(g.sort_values("delta_oof", ascending=False).drop(columns="rodada").round(4))
_ = salvar()

# 2. O conjunto escolhido no placar

Ele é medido sozinho (`f6_sel`) e com a média de 3 redes por dobra (`f6_sel_3redes`). Os vereditos são contra o `o5_top3` (`vs_base`), contra o melhor FT até agora (`o5_top1_3redes`, `vs_melhor`) e contra o trial 0. A última tabela compara com o MLP adotado.

In [ ]:
T0, B, MELHOR = ("ft_transformer", "trial0"), ("ft_transformer", F6.ROT_BASE), ("ft_transformer", "o5_top1_3redes")
contra = {"vs_trial0": T0, "vs_base": B, **({"vs_melhor": MELHOR} if "o5_top1_3redes" in anteriores else {})}
itens = [("trial0", F1.TRIAL0, None), (F6.ROT_BASE, F6.BASE, {"vs_trial0": T0})]
if escolhidas:
    c6 = F6.com_features(F6.BASE, escolhidas)
    itens += [("f6_sel", c6, contra), ("f6_sel_3redes", dict(hp=dict(c6["hp"], n_redes=3), dados=dict(c6["dados"])), contra)]
    print("Conjunto:", c6["dados"]["features"])
else:
    print("Nenhuma candidata passou na seleção: nada novo vai para o placar.")
linhas6 = medir(itens)
display(F1.tabela(linhas6, ("vs_trial0", "vs_base", "vs_melhor")).round(4))

RES = os.path.join(REPO_DIR, "resultados")
rot_mlp = json.load(open(os.path.join(RES, "escolhidos.json")))["mlp"]["rotulo"]
mlp = {r["rotulo"]: r for r in comum.ler_placar(RES) if r["modelo"] == "mlp"}[rot_mlp]
oof_mlp = np.load(os.path.join(RES, mlp["oof"]))
vv = lambda v: f"{v['veredito']} {v['delta']:+.4f} [{v['ic95'][0]:+.4f}, {v['ic95'][1]:+.4f}]"
comp = [dict(versao=r["rotulo"], ks_kfold=r["ks_kfold"], ks_oof=r["ks_oof"],
             vs_mlp=vv(comum.comparar_placar(y_tr, oof_mlp, np.load(os.path.join(PASTA, r["oof"])))))
        for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer" and r["rotulo"] in ("o5_top3", "o5_top1_3redes", "f6_sel", "f6_sel_3redes")]
print(f"Contra o MLP adotado ({rot_mlp}, KS do placar {mlp['ks_kfold']:.4f})")
display(pd.DataFrame(comp).round(4))
_ = salvar()

# 3. Placar das fases 1 a 6

In [ ]:
todas = [r for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer"]
tab = F1.tabela(sorted(todas, key=lambda r: -r["ks_kfold"]), ("vs_trial0", "vs_base", "vs_melhor"))
display(tab.round(4))
tab.to_csv(os.path.join(PASTA, "placar_ft_transformer_fase6.csv"), index=False)
open(os.path.join(PASTA, "PLACAR.md"), "w", encoding="utf-8").write(comum.relatorio_placar(PASTA))
ganharam = [r["rotulo"] for r in todas if (r.get("vs_trial0") or {}).get("veredito") == "ganhou"]
perderam = [r["rotulo"] for r in todas if (r.get("vs_trial0") or {}).get("veredito") == "perdeu"]
print(f"Ganharam do trial 0: {ganharam or 'nenhuma'}\nPerderam: {perderam or 'nenhuma'}")
salvar()
if wandb_run is not None:
    wandb_run.summary.update(dict(ganharam=ganharam, perderam=perderam))
    wandb_run.finish()
print(f"Tempo total: {(time.time() - INICIO) / 3600:.2f} h")
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))

### 📝 Análise da fase 6

- **Quais features entraram, e em que rodada?** _…_
- **O conjunto ganhou do `o5_top3` e do melhor FT no placar?** _…_
- **A média de 3 redes somou com as features?** _…_
- **E contra o MLP?** _…_
- **Próximo passo** (as mesmas features no MLP, adotar e medir o teste): _…_